# RAG Evaluation Metrics

## What Are We Evaluating?

RAG has **two components** — both must be evaluated independently:

```
Query → [Retriever] → docs → [LLM Generator] → Answer

         ↑                          ↑
  Retrieval Metrics          Generation Metrics
  Did we fetch right docs?   Did LLM answer correctly?
```

## The 6 Core Metrics at a Glance

| # | Metric | Evaluates | Needs Ground Truth? |
|---|---|---|---|
| 1 | **Context Precision** | Retriever — too much noise? | Yes |
| 2 | **Context Recall** | Retriever — missing anything? | Yes |
| 3 | **Context Relevance** | Chunk quality | No (LLM judge) |
| 4 | **Faithfulness** | Hallucination detection | No (LLM judge) |
| 5 | **Answer Relevance** | On-topic answer? | No (LLM judge) |
| 6 | **Answer Correctness** | Factually correct? | Yes |

## RAG Pipeline — Where Each Metric Applies

```
User Query
    ↓
VectorDB retrieval  ← Context Precision, Context Recall, Context Relevance
    ↓
Top-K docs
    ↓
LLM generates answer ← Faithfulness, Answer Relevance, Answer Correctness
    ↓
Final Answer
```

## Step 1 — Install Dependencies

In [ ]:
!pip install ragas langchain-openai chromadb datasets sentence-transformers

## Step 2 — Sample RAG Test Data (Medical Use Case)

To evaluate a RAG system you need:
- **question** — what the user asked
- **answer** — what your RAG system returned
- **contexts** — the documents retrieved from VectorDB
- **ground_truth** — the correct answer (for metrics that need it)

In [ ]:
from datasets import Dataset

# ── Test dataset: 3 question-answer pairs ──────────────────────────────────
# Each entry simulates what a RAG system produced for a patient query

rag_test_data = {
    "question": [
        "What is John's diabetes medication and dosage?",
        "What is John's HbA1c trend?",
        "Are there any drug interactions for John's medications?",
    ],
    "answer": [
        # RAG system answers (some good, some with issues — to test metrics)
        "John takes Metformin 500mg twice daily for Type 2 Diabetes since 2024.",
        "John's HbA1c was 8.2% in Jan 2024, then improved to 7.9% in Jun 2024.",   # missing Dec 2024
        "John takes Metformin 1000mg and Amlodipine — no interactions found.",      # HALLUCINATION (wrong dose + wrong interaction status)
    ],
    "contexts": [
        # Documents retrieved from VectorDB for each question
        [
            "John current medications: Amlodipine 5mg (since Jan 2026), Metformin 500mg (since 2024).",
            "John takes Metformin 500mg twice daily since 2024 for Type 2 Diabetes.",
            "Diabetes first-line treatment 2026: Metformin 500mg BID. Target HbA1c < 7.0%",
        ],
        
        [
            "John HbA1c trend: Jan 2024: 8.2%, Jun 2024: 7.9%, Dec 2024: 8.1%.",
            "HbA1c target for most Type 2 diabetic patients is less than 7.0%.",
        ],
        [
            "John current medications: Amlodipine 5mg (since Jan 2026), Metformin 500mg (since 2024).",
            "Amlodipine + Metformin: No major interaction. Monitor potassium. Safe combination.",
            "The weather in London is cloudy today.",   # ← irrelevant doc (tests precision)
        ],
    ],
    "ground_truth": [
        # Correct answers (used by metrics that need ground truth)
        "John takes Metformin 500mg twice daily since 2024 for Type 2 Diabetes.",
        "John's HbA1c trend: Jan 2024: 8.2%, Jun 2024: 7.9%, Dec 2024: 8.1% (slightly worsened).",
        "John takes Metformin 500mg and Amlodipine 5mg. No major interaction, but potassium should be monitored.",
    ]
}

dataset = Dataset.from_dict(rag_test_data)
print(f"Test dataset: {len(dataset)} question-answer pairs")
print(f"Columns: {dataset.column_names}")
print()

# Show what we're evaluating
for i in range(len(rag_test_data["question"])):
    print(f"Q{i+1}: {rag_test_data['question'][i]}")
    print(f"  Answer    : {rag_test_data['answer'][i][:70]}...")
    print(f"  Contexts  : {len(rag_test_data['contexts'][i])} docs retrieved")
    print(f"  Truth     : {rag_test_data['ground_truth'][i][:70]}...")
    print()

## Retrieval Metric 1 — Context Precision

> Of the documents retrieved, what fraction were actually relevant?

$$\text{Context Precision} = \frac{\text{Relevant docs in retrieved set}}{\text{Total docs retrieved}}$$

```
Q3 retrieved 3 docs:
  Doc 1: "John medications: Metformin 500mg, Amlodipine 5mg"  ← relevant ✅
  Doc 2: "Amlodipine + Metformin: no major interaction"       ← relevant ✅
  Doc 3: "Weather in London is cloudy today"                  ← NOT relevant ❌

Context Precision = 2/3 = 0.67
```

**Low precision** = LLM gets noisy, irrelevant context → worse answers.  
**Fix**: Better chunking, stricter similarity threshold, or add a reranker.

In [ ]:
def compute_context_precision(retrieved_docs: list, ground_truth_answer: str, threshold: float = 0.3) -> float:
    """
    Simulated context precision:
    Check each retrieved doc for keyword overlap with ground truth.
    In production RAGAS uses LLM-as-judge or embedding similarity.
    """
    gt_words = set(ground_truth_answer.lower().split())
    relevant_count = 0

    print(f"Ground Truth: {ground_truth_answer[:70]}...")
    print(f"Retrieved {len(retrieved_docs)} docs:\n")

    for i, doc in enumerate(retrieved_docs):
        doc_words   = set(doc.lower().split())
        overlap     = len(gt_words & doc_words) / max(len(gt_words), 1)
        is_relevant = overlap >= threshold
        relevant_count += int(is_relevant)
        status = "✅ relevant" if is_relevant else "❌ irrelevant"
        print(f"  Doc {i+1} [{status}] overlap={overlap:.2f}")
        print(f"         {doc[:70]}...")

    precision = relevant_count / len(retrieved_docs)
    print(f"\nContext Precision = {relevant_count}/{len(retrieved_docs)} = {precision:.2f}")
    return precision


print("=" * 60)
print("CONTEXT PRECISION — Q3 (Drug Interaction)")
print("=" * 60)
q3_precision = compute_context_precision(
    retrieved_docs   = rag_test_data["contexts"][2],
    ground_truth_answer = rag_test_data["ground_truth"][2]
)

print("\n" + "=" * 60)
print("CONTEXT PRECISION — Q1 (Medication)")
print("=" * 60)
q1_precision = compute_context_precision(
    retrieved_docs   = rag_test_data["contexts"][0],
    ground_truth_answer = rag_test_data["ground_truth"][0]
)

## Retrieval Metric 2 — Context Recall

> Of all the facts in the ground truth answer, how many were covered by the retrieved context?

$$\text{Context Recall} = \frac{\text{Ground truth facts covered by contexts}}{\text{Total facts in ground truth}}$$

```
Ground truth for Q2: "Jan 2024: 8.2%, Jun 2024: 7.9%, Dec 2024: 8.1%"
                                                         ↑
                                               this fact missing in answer!

Retrieved contexts covered: Jan 2024 ✅, Jun 2024 ✅, Dec 2024 ✅ (all in VectorDB)
Context Recall = 3/3 = 1.0  ← context was complete

But the RAG answer only mentioned Jan + Jun — that's an Answer problem, not Recall.
```

**Low recall** = VectorDB missed critical facts → LLM can't mention what it never saw.  
**Fix**: Retrieve more candidates (larger `top_k`), better chunking strategy.

In [ ]:
def compute_context_recall(retrieved_docs: list, ground_truth: str) -> float:
    """
    Simulated context recall:
    Check what fraction of ground truth key phrases appear in retrieved contexts.
    In production RAGAS uses LLM to extract claims then check each against contexts.
    """
    # Extract key phrases from ground truth (simplified — split by punctuation)
    import re
    # Split ground truth into atomic facts
    facts = [f.strip() for f in re.split(r'[,.]', ground_truth) if len(f.strip()) > 5]
    all_context = " ".join(retrieved_docs).lower()

    covered = 0
    print(f"Ground Truth Facts ({len(facts)}):")
    for fact in facts:
        # Check if key words from fact appear in any context
        fact_words     = set(fact.lower().split()) - {"the", "a", "an", "is", "was", "in", "for", "of"}
        words_found    = sum(1 for w in fact_words if w in all_context)
        fact_covered   = words_found / max(len(fact_words), 1) >= 0.6
        covered       += int(fact_covered)
        status = "✅" if fact_covered else "❌"
        print(f"  {status} '{fact.strip()[:60]}'")

    recall = covered / max(len(facts), 1)
    print(f"\nContext Recall = {covered}/{len(facts)} = {recall:.2f}")
    return recall


print("=" * 60)
print("CONTEXT RECALL — Q2 (HbA1c Trend)")
print("=" * 60)
q2_recall = compute_context_recall(
    retrieved_docs = rag_test_data["contexts"][1],
    ground_truth   = rag_test_data["ground_truth"][1]
)

print("\n" + "=" * 60)
print("CONTEXT RECALL — Q1 (Medication)")
print("=" * 60)
q1_recall = compute_context_recall(
    retrieved_docs = rag_test_data["contexts"][0],
    ground_truth   = rag_test_data["ground_truth"][0]
)

## Generation Metric 3 — Faithfulness (Most Important!)

> Is every claim in the LLM's answer actually supported by the retrieved context?  
> **Detects hallucination.**

```
Answer claim: "John takes Metformin 1000mg"
Context says: "John takes Metformin 500mg"
→ CONTRADICTION → Faithfulness score drops

Answer claim: "No interactions found"
Context says: "Monitor potassium levels"
→ CONTRADICTION → Another hallucination detected

Faithfulness = supported_claims / total_claims
```

**Low faithfulness** = LLM is making up facts not in the context = HALLUCINATION.  
**Fix**: Stricter prompt ("only answer from provided context"), better context quality.

In [ ]:
def compute_faithfulness(answer: str, contexts: list) -> float:
    """
    Simulated faithfulness check:
    Split answer into claims, check each claim against context.
    In production RAGAS uses LLM to extract claims + verify each against context.
    """
    import re
    # Break answer into individual claims
    claims = [c.strip() for c in re.split(r'[.,]', answer) if len(c.strip()) > 8]
    all_context = " ".join(contexts).lower()

    supported = 0
    print(f"Answer: '{answer}'")
    print(f"\nClaims extracted ({len(claims)}):")

    for claim in claims:
        claim_words = set(claim.lower().split()) - {"the", "a", "an", "is", "was", "in", "for", "of", "and"}
        words_in_ctx = sum(1 for w in claim_words if w in all_context)
        overlap = words_in_ctx / max(len(claim_words), 1)
        is_supported = overlap >= 0.55
        supported += int(is_supported)
        status = "✅ supported" if is_supported else "🚨 HALLUCINATION"
        print(f"  {status} (overlap={overlap:.2f}): '{claim.strip()[:65]}'")

    faithfulness = supported / max(len(claims), 1)
    print(f"\nFaithfulness = {supported}/{len(claims)} = {faithfulness:.2f}")
    return faithfulness


print("=" * 60)
print("FAITHFULNESS — Q1 (Good answer)")
print("=" * 60)
f1 = compute_faithfulness(rag_test_data["answer"][0], rag_test_data["contexts"][0])

print("\n" + "=" * 60)
print("FAITHFULNESS — Q3 (Hallucinated answer)")
print("=" * 60)
f3 = compute_faithfulness(rag_test_data["answer"][2], rag_test_data["contexts"][2])

## Generation Metric 4 — Answer Relevance

> Does the answer actually address what was asked?

```
Query:  "What is John's HbA1c trend?"
Answer: "Diabetes is a metabolic disorder affecting insulin production..."
→ Talks about diabetes in general, NOT John's HbA1c → relevance = 0.1

Answer: "John's HbA1c was 8.2% Jan 2024, improved to 7.9% Jun 2024"
→ Directly answers the question → relevance = 0.95
```

**Low answer relevance** = LLM went off-topic, over-explained, or answered a different question.  
**Fix**: Better system prompt, more specific query formulation.

In [ ]:
def compute_answer_relevance(question: str, answer: str) -> float:
    """
    Simulated answer relevance:
    Check keyword overlap between question and answer.
    In production RAGAS generates synthetic questions from the answer
    and measures cosine similarity with original question using embeddings.
    """
    q_words = set(question.lower().split()) - {"what", "is", "are", "the", "a", "an", "for", "of"}
    a_words = set(answer.lower().split())
    overlap = len(q_words & a_words) / max(len(q_words), 1)

    print(f"Question: '{question}'")
    print(f"Answer  : '{answer[:80]}...'")
    print(f"\nQuestion keywords : {q_words}")
    print(f"Overlap with answer: {len(q_words & a_words)}/{len(q_words)} = {overlap:.2f}")

    # Penalize very long answers (sign of going off-topic)
    length_penalty = min(1.0, 30 / max(len(answer.split()), 1))
    final_score = overlap * (0.7 + 0.3 * length_penalty)
    print(f"Answer Relevance = {final_score:.2f}")
    return final_score


print("=" * 60)
print("ANSWER RELEVANCE — Q1 (On-topic answer)")
print("=" * 60)
ar1 = compute_answer_relevance(rag_test_data["question"][0], rag_test_data["answer"][0])

print("\n" + "=" * 60)
print("ANSWER RELEVANCE — Off-topic answer (demo)")
print("=" * 60)
off_topic = "Diabetes mellitus is a group of metabolic diseases characterized by hyperglycemia resulting from defects in insulin secretion, insulin action, or both."
ar_bad = compute_answer_relevance(rag_test_data["question"][0], off_topic)

## Generation Metric 5 — Answer Correctness

> Is the answer factually correct compared to the ground truth?

$$\text{Answer Correctness} = \frac{\text{Correct facts in answer}}{\text{Total facts in ground truth}}$$

```
Ground truth: "Metformin 500mg twice daily since 2024"
Answer A:     "Metformin 500mg twice daily since 2024"  → correctness = 1.0
Answer B:     "Metformin 250mg once daily"              → correctness = 0.3  (wrong dose + frequency)
Answer C:     "Metformin 1000mg"                        → correctness = 0.4  (wrong dose)
```

**Requires labeled ground truth** — unlike faithfulness/relevance which use LLM-as-judge.  
**Use when**: You have a test set with known correct answers (e.g., from expert annotators).

In [ ]:
def compute_answer_correctness(answer: str, ground_truth: str) -> float:
    """
    Simulated answer correctness using token F1 (like SQuAD metric).
    In production RAGAS uses LLM to compare answer vs ground truth semantically.
    """
    def tokenize(text):
        return set(text.lower().split()) - {"the", "a", "an", "is", "was", "in", "for", "of", "and", "s"}

    answer_tokens = tokenize(answer)
    truth_tokens  = tokenize(ground_truth)

    common  = answer_tokens & truth_tokens
    precision = len(common) / max(len(answer_tokens), 1)
    recall    = len(common) / max(len(truth_tokens), 1)
    f1        = 2 * precision * recall / max(precision + recall, 1e-8)

    print(f"Answer      : '{answer[:70]}...'")
    print(f"Ground Truth: '{ground_truth[:70]}...'")
    print(f"\nToken overlap: {len(common)} common tokens")
    print(f"  Precision = {precision:.2f} | Recall = {recall:.2f}")
    print(f"  Answer Correctness (F1) = {f1:.2f}")
    return f1


print("=" * 60)
print("ANSWER CORRECTNESS — Q1 (Correct answer)")
print("=" * 60)
ac1 = compute_answer_correctness(rag_test_data["answer"][0], rag_test_data["ground_truth"][0])

print("\n" + "=" * 60)
print("ANSWER CORRECTNESS — Q3 (Hallucinated answer)")
print("=" * 60)
ac3 = compute_answer_correctness(rag_test_data["answer"][2], rag_test_data["ground_truth"][2])

## Full Evaluation — RAGAS Library (Production Way)

RAGAS automates all metrics with LLM-as-judge. One `evaluate()` call scores everything.

In [ ]:
import os
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_correctness,
)
from datasets import Dataset

# Set your OpenAI API key (RAGAS uses LLM-as-judge internally)
# os.environ["OPENAI_API_KEY"] = "sk-..."  # uncomment and set your key

# Same test data as above — RAGAS expects this exact schema
dataset = Dataset.from_dict(rag_test_data)

# ── Run full RAGAS evaluation ─────────────────────────────────────────────
# Requires OPENAI_API_KEY — uncomment when key is available
# result = evaluate(
#     dataset,
#     metrics=[
#         faithfulness,        # hallucination check (no ground truth needed)
#         answer_relevancy,    # on-topic check (no ground truth needed)
#         context_precision,   # retrieval noise check (needs ground truth)
#         context_recall,      # retrieval coverage check (needs ground truth)
#         answer_correctness,  # factual accuracy (needs ground truth)
#     ]
# )
# print(result)
# result.to_pandas()  # converts to DataFrame for analysis

# ── Simulated RAGAS output for demonstration ─────────────────────────────
import pandas as pd

simulated_results = pd.DataFrame({
    "question"          : rag_test_data["question"],
    "faithfulness"      : [0.97,  0.85,  0.12],   # Q3 hallucinates → very low
    "answer_relevancy"  : [0.95,  0.92,  0.88],   # all somewhat on-topic
    "context_precision" : [0.95,  1.00,  0.67],   # Q3 has irrelevant doc
    "context_recall"    : [1.00,  1.00,  0.90],   # contexts mostly complete
    "answer_correctness": [0.96,  0.78,  0.21],   # Q3 factually wrong
})

print("RAGAS Evaluation Results (Simulated)")
print("=" * 80)
print(simulated_results.to_string(index=False))

print("\n\nAverage Scores:")
numeric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "answer_correctness"]
for col in numeric_cols:
    avg = simulated_results[col].mean()
    status = "✅" if avg >= 0.8 else "⚠️ " if avg >= 0.6 else "🚨"
    print(f"  {status} {col:25s}: {avg:.2f}")

## Summary — Complete Reference

## All 6 Metrics

| # | Metric | Evaluates | Needs Ground Truth? | Low Score Means | Fix |
|---|---|---|---|---|---|
| 1 | **Context Precision** | Retriever noise | Yes | Too many irrelevant docs fetched | Add reranker, stricter threshold |
| 2 | **Context Recall** | Retriever coverage | Yes | Missing relevant docs | Increase top_k, better chunking |
| 3 | **Context Relevance** | Chunk quality | No (LLM judge) | Chunks are off-topic | Better chunking strategy |
| 4 | **Faithfulness** | Hallucination | No (LLM judge) | LLM invents facts | Stricter prompt, better context |
| 5 | **Answer Relevance** | On-topic answer | No (LLM judge) | LLM went off-topic | Better system prompt |
| 6 | **Answer Correctness** | Factual accuracy | Yes (ground truth) | Factually wrong answer | Improve retrieval + generation |

## When to Use Which Metric

| Scenario | Primary Metric | Why |
|---|---|---|
| LLM making up facts | **Faithfulness** | Hallucination detection — always monitor |
| Wrong docs retrieved | **Context Precision** | Garbage in = garbage out |
| LLM missing key facts | **Context Recall** | VectorDB not finding relevant docs |
| Answer going off-topic | **Answer Relevance** | LLM ignoring the question |
| Comparing two RAG versions | **All 5 RAGAS metrics** | Full A/B comparison |
| No ground truth available | Faithfulness + Answer Relevance | LLM-as-judge, no labels needed |
| Have labeled test set | Answer Correctness + Recall | Precise accuracy measurement |

## Priority Order

```
1. Faithfulness        ← MUST be high. Hallucinations destroy trust.
2. Context Precision   ← Noise in context → noise in answer.
3. Answer Relevance    ← Did we actually answer the question?
4. Context Recall      ← Are we missing key information?
5. Answer Correctness  ← Final factual accuracy check (needs labels).
```

## Vectorless RAG vs Vector RAG — Metric Behavior

| Metric | Vector RAG | BM25/Keyword RAG |
|---|---|---|
| Context Precision | Better on semantic queries | Better on exact keyword queries |
| Context Recall | Better for paraphrases/synonyms | Misses synonyms ("MI" ≠ "heart attack") |
| Faithfulness | Same — depends on LLM | Same |
| Answer Relevance | Same — depends on LLM | Same |